<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

----------

<h1 align="left"> Building a Simple RAG System from Scratch </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

----------

In [ ]:
!pip install faiss-cpu sentence-transformers transformers datasets==2.14.5

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.1 MB/

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import pandas as pd

**SAMPLE DOCUMENTS**

In [ ]:
documents = [ "The capital of Canada is Ottawa.", "Python is a popular programming language for AI and machine learning.", "The Great Wall of China is visible from space is a myth.", "FAISS is a library for efficient similarity search and clustering of dense vectors.", "Transformers models are widely used for natural language processing tasks." ]

**EMBEDDING MODEL**

In [ ]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 **CREATE FAISS INDEX**


In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print(f"FAISS index contains {index.ntotal} documents.")


FAISS index contains 5 documents.


**RETRIEVAL FUNCTION**

In [ ]:
def retrieve(query, top_k=2):
    query_emb = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_emb, top_k)
    retrieved_docs = [documents[i] for i in indices[0]]
    return retrieved_docs

**GENERATION MODEL (QA or Summarization)**

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


**RAG PIPELINE FUNCTION**

In [ ]:
def rag_pipeline(query):
    retrieved_docs = retrieve(query)
    context = " ".join(retrieved_docs)
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"
    response = generator(prompt, max_length=100, temperature=0.3)[0]['generated_text']

    print("🔍 Retrieved Documents:\n", retrieved_docs)
    print("\n💬 Model Answer:\n", response)

**TEST THE PIPELINE**

In [ ]:
rag_pipeline("What is FAISS used for?")
rag_pipeline("Where is the capital of Canada?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 Retrieved Documents:
 ['FAISS is a library for efficient similarity search and clustering of dense vectors.', 'Python is a popular programming language for AI and machine learning.']

💬 Model Answer:
 efficient similarity search and clustering of dense vectors


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 Retrieved Documents:
 ['The capital of Canada is Ottawa.', 'The Great Wall of China is visible from space is a myth.']

💬 Model Answer:
 Ottawa
